# Validation and monitoring

Research question: What fails explicitly, and what survives a source failure?

Inputs: frozen Treasury 2024–2025 CSVs. Rates are decimals, prices per 100 face. Release 1.0.0. No live signal.

In [1]:
import sys,json
from pathlib import Path
from datetime import date
import numpy as np
R=Path.cwd()
if R.name=='notebooks':R=R.parent
sys.path.insert(0,str(R/'src'));sys.path.insert(0,str(R/'scripts'))
from curve_lab.data import load_history
from curve_lab.bonds import Bond
from curve_lab.curves import bootstrap,par_rate,TENORS,KEYS,fit_nelson_siegel
from curve_lab.risk import bond_risk,hedge,carry_bridge
from curve_lab.forecast import predict,evaluate
from review_display import table,figure
if 'display_bundle' not in globals():
    from IPython.display import display
    def display_bundle(bundle):display(bundle,raw=True)
D=json.loads((R/'artifacts/release/research.json').read_text());rows,receipts=load_history(R)
print('Run',D['run_id'],'| source through',D['as_of'],'| hypothetical bond valuations')


Run p4-662f58a64e8e | source through 2025-12-31 | hypothetical bond valuations


## Audit all historical calibration identities

In [2]:
errs=[]
for row in rows:
 c=bootstrap(row['par']);errs.extend(abs(par_rate(c,t)-p)*10000 for t,p in zip(TENORS,row['par']))
print('Curves',len(rows),'input repricing checks',len(errs),'max error bp',max(errs));assert max(errs)<1e-8

Curves 499 input repricing checks 4491 max error bp 3.1225022567582528e-12


## Challenge invalid data and unsupported tenor

In [3]:
failures=[]
for name,fn in [('missing tenor',lambda:bootstrap([float('nan')]*9)),('extrapolation',lambda:bootstrap(rows[-1]['par']).df(31)),('settle at maturity',lambda:Bond(date(2030,1,1),.04).cashflows(date(2030,1,1)))]:
 try:fn()
 except ValueError:failures.append(name)
assert len(failures)==3;display_bundle(table(['Rejected case'],[[x] for x in failures]))

Rejected case
missing tenor
extrapolation
settle at maturity


## Read release and operational boundaries

In [4]:
print('Research is frozen:',D['as_of']);print('Market refresh is separate; failed refresh retains last-known-good.');print('Source vintages absent; native Jupyter kernel not claimed by sequential executor.');display_bundle(table(['Source','SHA-256'],[[r['path'],r['sha256']] for r in receipts]))

Research is frozen: 2025-12-31
Market refresh is separate; failed refresh retains last-known-good.
Source vintages absent; native Jupyter kernel not claimed by sequential executor.


Source,SHA-256
data/raw/treasury_par_2024.csv,6775840b76cd7fd41121f61d6f6c60184eca6b2ba49954ec7372c463cbeef34d
data/raw/treasury_par_2025.csv,cf2fa3da7f160384b63d2ead698532b1d76cd9f789ce4911e2304db85b14e6d5


## Interpretation

Validation confirms stated mathematical and failure contracts. Source correctness, historical availability and actual-security execution remain separate evidence requirements. See docs/runbook.md for refresh and rollback steps.

Canonical evidence: `artifacts/release/research.json`; source receipts: `data/manifests/sources.json`.